# The Lace agent's graph

A Lace is one ReAct agent per startup. Its LangGraph graph is built and compiled by
`build_lace_graph()` in `agents/lace/graph.py` (`builder.compile(checkpointer=...)`):

- `agent`: the model call, with the system prompt (what the Lace knows, what is on file) and the tools it may use this turn.
- `tools`: a `ToolNode` that runs whatever tools the model asked for, then hands back to `agent`.
- `agent` routes to `tools` while the model asks for tools, else ends the turn.

Building the graph needs no database or model: the store and the model are only used when it runs,
so placeholders are enough to draw it.

In [ ]:
import sys
from pathlib import Path

# Walk up from cwd to find project root (has pyproject.toml)
root = Path.cwd()
while root != root.parent:
    if (root / "pyproject.toml").exists():
        break
    root = root.parent

sys.path.insert(0, str(root))

from IPython.display import Image, Markdown, display

from agents.lace.graph import build_lace_graph
from agents.lace.tools import build_tools
from config import get_settings

In [ ]:
class Placeholder:
    """Stands in for the Lace's store and job runner: drawing the graph never calls them."""

    def __getattr__(self, name):
        raise RuntimeError(f"{name} is not available when only drawing the graph")


# With `jobs`, the graph also gets the tools that start background work (searches, drafting).
graph = build_lace_graph(Placeholder(), jobs=Placeholder())
drawable = graph.get_graph()

In [ ]:
# The diagram as a PNG (rendered by mermaid.ink, so it needs a network connection);
# without one, the Mermaid source below draws the same thing.
try:
    display(Image(drawable.draw_mermaid_png()))
except Exception as exc:  # noqa: BLE001 -- the source below is the fallback
    print(f"Could not render the PNG ({type(exc).__name__}); use the Mermaid source below.")

In [ ]:
print(drawable.draw_mermaid())

## The tools behind the `tools` node

Everything the Lace can do in a turn. The ones that start background work only appear when the
graph is built with `jobs` (as the backend does).

In [ ]:
tools = build_tools(Placeholder(), get_settings(), jobs=Placeholder())
display(Markdown("\n".join(f"- `{t.name}`: {t.description.splitlines()[0]}" for t in tools)))